### G. ????????? ???? ????? ??????? (30 ??????)

### ???????????
- **?????:** 10 ??????
- **??????:** 244.14 ??
- **????:** CSV-???? ? ???????
- **?????:** `output.txt`

### ???????? ??????
---

?????? ? IT ????????? ??? ????? ???????????. ?????????? ????????, ???????? ????? ? ????? ? ??????????????? ????????? ? ??? ??? ????????? ?????? ?? ????????? ? ????????????? ???? ??? ?????? ???????. ?? ???? ???? ??-?????????? ?????????? ???: **???? ??????? ????**.

?????? ???????? ????????: ???????????? ???????? ?????????, ???????? ???????, ????????? ???????, ?????????, ????????? ?????? ? ??????????? ?? ???????. ?????? ?????? ????? ???????? ?????: **?????? ?? ?? ?????????? ??????????, ????? ??????????????? ??????? ? ??????? ??????**

??? ????? ???? ?????? ???????? ??????? history.csv ? ???????????????? ??????????? ?? ??? ????? ?????? ?? **?????? 2025 ????**. ?? ?????? ???? ?????? ??? ????? ??????:

* ????? ?????? ????? **??????? ? ??? 2025 ????**;
* ??????? ????? ?? ??????? ? ???, ???????:

  * ??????? ?? **????????????? ???????**;
  * ??????? ?? **???????**.

---

### ?? ?????? ?????

CSV-???? ???????? ????????? ??????????:

| ????                        | ????????                                                                   |
| --------------------------- | -------------------------------------------------------------------------- |
| `id`                        | ?????????? ????????????? ??????                                            |
| `logins_current_month`      | ?????????? ????, ????? ????? ??????? ? ???? ?? ????????? ??????????? ????? |
| `country`                   | ?????? ??????                                                              |
| `traffic_type`              | ???????? ??????? (???????????? ??? ?????????)                              |
| `platform`                  | ????????? (Android ??? iOS)                                                |
| `registration_date`         | ???? ???????????                                                           |
| `ads_shown_current_month`   | ???-?? ??????? ??????? ?? ????????? ?????                                  |
| `ads_revenue_current_month` | ??????? ?? ??????? ?? ????????? ?????                                      |
| `ads_shown_current_day`     | ??????? ??????? ? ????????? ??????? ????                                   |
| `ads_revenue_current_day`   | ??????? ?? ??????? ? ????????? ????                                        |
| `revenue_current_month`     | ??????? ?? ????????????? ??????? ?? ?????                                  |
| `games_pvp`                 | ???-?? PvP-???????? ?? ?????                                               |
| `wins_pvp`                  | ?????? ? PvP ?? ?????                                                      |
| `quests`                    | ??????????? ??????? ?? ?????                                               |
| `hard_quests`               | ??????? ?????? ?? ?????                                                    |
| `current_passed_level`      | ????????? ?????????? ???????                                               |
| `offers`                    | ???-?? ???????????                                                         |
| `currency_i`                | ????????????? ?????? ?i                                                    |
| `current_avg_ping`          | ??????? ???? ??????                                                        |
| `current_month`             | ?????, ? ???????? ????????? ??????                                         |

---

### ?? ?????? ??????

???????? ???? ? CSV ? **????? ?????????**:

| ???????              | ????????                                                                                   |
| -------------------- | ------------------------------------------------------------------------------------------ |
| `id`                 | ????????????? ????????????                                                                 |
| `revenue_next_month` | ????????????? ??????? ?? ???????????? ? ??? (??????? + ???????), **??????????????? ?????** |
| `is_active`          | `1`, ???? ????? **????? ??????? ? ???** (?????? ???? ?? ???); ????? `0`                    |

---

### ?? ??????? ? ??????

#### ????????????? (`is_active`)

??????????? \$F\_1\$-???? **???????????? ?????????????? ??????** (\$1\$).

#### ????????? (`revenue_next_month`)

????????? **log-RMSE** ? ??????????????? ?????????????????? ?????? ????? ???????? ?????????????:

```python
import numpy as np
from sklearn.metrics import mean_squared_error

def log_rmse(y_true, y_pred):
    return mean_squared_error(np.log1p(y_true), np.log1p(y_pred), squared=False)
```

#### ????????? ????

????? ?????? ?????????????? ?? ???????:

$$
\text{Score} = 15 \cdot F_1 + \max(15 - 20 \cdot \text{log\_rmse}, 0)
$$

---

?? ???????????? ????: **30**
? ?? **15 ??????** ?? ?????????????
? ?? **15 ??????** ?? ?????????

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

## Classiication

In [ ]:
df = pd.read_csv('history.csv')

In [ ]:
df["traffic_type"]

In [ ]:
"""
#traffic_type
#platform
registration_date
current_month
"""

In [ ]:
df = df.sort_values('id')
df["IOS"] = df["platform"].apply(lambda x: 1 if x == "iOS" else 0)
df["organic"] = df["traffic_type"].apply(lambda x: 1 if x == "organic" else 0)
df.drop(columns=["platform", "traffic_type"], inplace=True)

In [ ]:
df['current_month'] = pd.to_datetime(df['current_month'])
df['next_month'] = df['current_month'] + pd.DateOffset(months=1)
id_month_set = set(df[['id', 'current_month']].itertuples(index=False, name=None))

def check_online(row):
        return int((row['id'], row['next_month']) in id_month_set)

df['isOnlineNextMonth'] = df.apply(check_online, axis=1)


In [ ]:
df['registration_date'] = pd.to_datetime(df['registration_date'], format='%Y-%m-%d')
may_1_2025 = pd.to_datetime('2025-05-01')
df['days_since_registration'] = (may_1_2025 - df['registration_date']).dt.days
df.drop(columns=["registration_date"], inplace=True)

In [ ]:
df['isOnlineNextMonth_filled'] = df['isOnlineNextMonth'].fillna(-1)  # ????? None ?? ?????
df = df.sort_values(
    by=['id', 'isOnlineNextMonth_filled', 'current_month'],
    ascending=[True, False, False]
)

df = df.drop(columns=['isOnlineNextMonth_filled']).groupby('id', as_index=False).first()


In [ ]:
pos_df = df[df['isOnlineNextMonth'] == 1]
neg_df = df[df['isOnlineNextMonth'] == 0].sample(frac=1, random_state=42)
train_df = pd.concat([pos_df, neg_df], ignore_index=True)


In [ ]:
target = 'isOnlineNextMonth'
ignore_cols = ['id', 'current_month', 'next_month', target]

# ????????
X = train_df.drop(columns=ignore_cols)
y = train_df[target]

# ????????????? ????????? ?????????????? ???????? (???? ???? ?????????)
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='F1',
    verbose=100,
    random_seed=42
)

model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_test, y_test), early_stopping_rounds=50)

In [ ]:
y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
train_ids = set(train_df['id'])
full_ids = set(df['id'])
predict_ids = full_ids - train_ids

df_to_predict = df[df['id'].isin(predict_ids)].copy()
X_pred = df_to_predict.drop(columns=['id', 'current_month', 'isOnlineNextMonth'], errors='ignore')
df_to_predict['is_active'] = model.predict(X_pred)

df_train_active = train_df.copy()
df_train_active['is_active'] = df_train_active['isOnlineNextMonth']

final_df = pd.concat([df_train_active, df_to_predict], ignore_index=True)

In [ ]:
final_df = final_df.sort_values('id')
final_df

## Regression

In [ ]:
ids_active = final_df.loc[final_df['is_active'] == 1, 'id']

In [ ]:
df = pd.read_csv('history.csv')

df = df.sort_values('id')
df["IOS"] = df["platform"].apply(lambda x: 1 if x == "iOS" else 0)
df["organic"] = df["traffic_type"].apply(lambda x: 1 if x == "organic" else 0)
df.drop(columns=["platform", "traffic_type"], inplace=True)

df['registration_date'] = pd.to_datetime(df['registration_date'], format='%Y-%m-%d')
may_1_2025 = pd.to_datetime('2025-05-01')
df['days_since_registration'] = (may_1_2025 - df['registration_date']).dt.days
df.drop(columns=["registration_date"], inplace=True)


In [ ]:
df['sum'] = df['revenue_current_month'] + df['ads_revenue_current_month']
df.drop(columns=['revenue_current_month', 'ads_revenue_current_month', 'ads_shown_current_month', 'ads_shown_current_day', 'ads_revenue_current_day'], inplace=True)

## Create submission

In [ ]:
df_s = pd.read_csv('history.csv')
df_s['is_active'] = 0
df_s['next_month_revenue'] = 0.0
df_sorted = df_s.sort_values('id', ascending=True)
df_unique = df_sorted.drop_duplicates(subset='id', keep='first')
submission = df_unique[['id', 'next_month_revenue', 'is_active']]


In [ ]:
final_df['id'] = final_df['id'].astype(str)
submission['id'] = submission['id'].astype(str)

id_to_active = dict(zip(final_df['id'], final_df['is_active']))
submission['is_active'] = submission['id'].map(id_to_active)

In [ ]:
submission

## Save submission

In [ ]:
submission.to_csv('submission.csv', index=False)